# Task 3 — ML Model Portfolio
**Module:** 7006SCN Machine Learning and Big Data  

---

## 3.1 Why These Four Models

The US airline flight delay dataset has a mixed structure; including scaled numerical predictors (`Distance`, `CRSElapsedTime`, `hour_of_day`), indexed categoricals (`airline_idx`, `origin_idx`, `dest_idx`) and temporal features (`Month`, `DayOfWeek`). The target is imbalanced with ~21% delayed by 15+ minutes, hence the choice to focus on AUC-ROC and AUC-PR metrics over the accuracy metric which would be inflated in favour of classifying to the "No Delay" category by a model like Logistic Regression if not carefully configured. It is for these reasons that 4 classifiers are selected to represent linear, probabilistic, bagging ensemble, and boosting style ensemble methods in PySpark MLlib.

The models are selected based on the premise they cover the landscape of different classification algorithms whilst also ensuring they can accommodate the types of data the dataset features, and in turn serve different roles in understanding predictive performance and interpretability.

**Logistic Regression** is the natural linear model for classification, with both numeric features appropriately scaled in the preceding Task 2 pipeline, making it a good linear baseline. Two values of `regParam` ([0.01, 0.1]) were chosen to experiment with light versus heavier L2 penalisation and two values of `elasticNetParam` ([0.0, 0.5]) were used to assess whether a simpler ridge regression is sufficient versus a mixed L1/L2 regularisation that encourages feature sparsity for the index variables — as practised in the Week 3 lab — to better potentially remove noise in those high-cardinality feature representations.

**Naive Bayes** assumes the independence of feature conditional distributions, which is not strictly true given that features like the month, hour of day, and distance of travel interact in influencing airline delays. Furthermore, Naive Bayes requires non-negative feature inputs. Therefore, rather than using features post-StandardScaler from Task 2, a new pipeline is constructed for Naive Bayes using raw numerical features that are intrinsically non-negative, following the Week 3 Day 2 pipeline methodology. Laplace smoothing is varied across [0.5, 1.0, 2.0] to assess whether the data benefits from this regularisation to address potential zero-count issues in less frequent categorical value combinations within the multinomial calculation.

**Random Forest** as an ensemble model can intrinsically capture non-linearities that the previous, relatively linear, models cannot. By bootstrapping rows and sampling features, RF aims to improve the generality of the ensemble members and reduce variance. A smaller random sampling fraction (30%) is used for training when CrossValidator is run, limiting potential driver memory issues under local Spark execution with the large 10M+ training row size while preserving the randomisation characteristics of the method. Tuning focuses on variations in both ensemble size (`numTrees`) and tree depth (`maxDepth`), aiming to avoid a simple single-tree solution while preventing excessive complexity. RF's feature importance outputs can also directly feed into the SHAP feature attribution analysis in Task 5.

**Gradient Boosted Trees** are ensemble models where trees are added in sequence to correct the errors made by prior trees, resulting in high predictive power when predictive signal is "spread thinly" across many low-informative features — as one might hypothesise for the multifactorial problem of flight delays. A conservative range of maximum iterations (`maxIter` [10, 20]) is considered along with smaller step sizes ([0.05, 0.1]) to probe the effects of more thorough training with smaller correction increments versus fewer iterations with larger increments. As with RF, training is restricted to a 30% random sample of the total training dataset for the same computational constraint reasons outlined above.

## 3.2 Tuning Approach

All four models were trained inside a `CrossValidator` with `ParamGridBuilder` (3 folds), using `BinaryClassificationEvaluator` with `areaUnderROC` as the optimisation metric. AUC-PR was also recorded separately because — as the Week 4 lab notes highlight — precision-recall curves are more informative than ROC curves under class imbalance. With a ~21% positive rate, AUC-PR provides a more demanding test of whether the model can successfully rank true delays above non-delays without being inflated by the large majority-class pool.

| Model | Params tuned | Search space | Folds |
|---|---|---|---|
| Logistic Regression | regParam, elasticNetParam | [0.01, 0.1] × [0.0, 0.5] | 3 |
| Naive Bayes | smoothing | [0.5, 1.0, 2.0] | 3 |
| Random Forest | numTrees, maxDepth | [10, 20] × [3, 5] | 3 |
| GBT | maxIter, stepSize | [10, 20] × [0.05, 0.1] | 3 |

---

In [1]:
# same environment setup as Task 1 and Task 2
import os, sys, time, pyspark
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

os.environ['JAVA_HOME'] = '/opt/anaconda3/envs/spark311'
os.environ['PATH']      = os.environ['JAVA_HOME'] + '/bin:' + os.environ.get('PATH', '')

spark = (
    SparkSession.builder
    .master('local[2]')
    .appName('7006SCN_Airline_Task3')
    .config('spark.driver.memory', '8g')
    .config('spark.sql.shuffle.partitions', '20')
    .config('spark.ui.showConsoleProgress', 'false')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

DATA_DIR  = Path('/tmp/airline')
PROCESSED = DATA_DIR / 'processed'
MODELS    = DATA_DIR / 'models'
MODELS.mkdir(parents=True, exist_ok=True)

print('Spark version:', spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/28 13:11:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1


In [2]:
# load scaled train/test splits saved by Task 2
from pyspark.storagelevel import StorageLevel

train_df = spark.read.parquet(str(PROCESSED / 'train'))
test_df  = spark.read.parquet(str(PROCESSED / 'test'))

train_df.persist(StorageLevel.DISK_ONLY)
test_df.persist(StorageLevel.DISK_ONLY)

print(f'Train rows : {train_df.count():,}')
print(f'Test rows  : {test_df.count():,}')
print(f'Columns    : {train_df.columns}')
train_df.show(3, truncate=80)

Train rows : 10,169,560
Test rows  : 2,544,963
Columns    : ['features', 'label']
+--------------------------------------------------------------------------------+-----+
|                                                                        features|label|
+--------------------------------------------------------------------------------+-----+
|[-1.5952182357191167,1.0099634087814486,1.4177799703872525,-0.006589901431013...|  0.0|
|[1.3148926130637597,0.5109479208378938,1.2144797233120201,0.3502611705714948,...|  0.0|
|[0.4418593584288967,0.011932432894338738,0.4012787350110901,-0.52814146820391...|  0.0|
+--------------------------------------------------------------------------------+-----+
only showing top 3 rows


In [3]:
# import everything needed for all four models
# CrossValidator + ParamGridBuilder pattern from Week 3 lab
from pyspark.ml.classification import (
    LogisticRegression, NaiveBayes,
    RandomForestClassifier, GBTClassifier
)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
import numpy as np, json

# evaluators reused for every model
auc_roc_eval = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC')
auc_pr_eval  = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderPR')
f1_eval      = MulticlassClassificationEvaluator(
    labelCol='label', predictionCol='prediction', metricName='f1'
)

results = {}
print('All imports done, evaluators ready')

All imports done, evaluators ready


## Model 1 — Logistic Regression

In [4]:
# Model 1: Logistic Regression
# linear classifier, fast to train, good baseline
# regParam and elasticNetParam grid same as Week 3 lab
print('Training Logistic Regression...')

lr = LogisticRegression(featuresCol='features', labelCol='label', maxIter=100)

lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam,        [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

lr_cv = CrossValidator(
    estimator=lr, estimatorParamMaps=lr_grid,
    evaluator=auc_roc_eval, numFolds=3, seed=42
)

t0 = time.time()
lr_model = lr_cv.fit(train_df)
lr_time  = time.time() - t0

lr_preds  = lr_model.transform(test_df)
lr_auc    = auc_roc_eval.evaluate(lr_preds)
lr_auc_pr = auc_pr_eval.evaluate(lr_preds)
lr_f1     = f1_eval.evaluate(lr_preds)

results['Logistic Regression'] = {
    'auc_roc': lr_auc, 'auc_pr': lr_auc_pr, 'f1': lr_f1, 'train_sec': lr_time
}

print(f'Train time : {lr_time:.1f}s')
print(f'AUC-ROC    : {lr_auc:.4f}')
print(f'AUC-PR     : {lr_auc_pr:.4f}')
print(f'F1         : {lr_f1:.4f}')

Training Logistic Regression...
Train time : 159.1s
AUC-ROC    : 0.6208
AUC-PR     : 0.2763
F1         : 0.6980


In [5]:
# LR confusion matrix - groupBy pattern from the lab
print('Logistic Regression — Confusion Matrix')
lr_preds.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()

lr_preds.select('label', 'prediction', 'probability') \
    .write.mode('overwrite').parquet(str(DATA_DIR / 'predictions_lr'))

Logistic Regression — Confusion Matrix
+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|  0.0|       0.0|2011868|
|  1.0|       0.0| 533095|
+-----+----------+-------+



## Model 2 — Naive Bayes

NB needs non-negative features. The StandardScaler output from Task 2 has negative values because `withMean=True` centres the data. So for NB I build a separate pipeline using only the raw positive numeric features, same approach as the Week 3 Day 2 lab.

In [6]:
# Model 2: Naive Bayes
# can't use scaled features (negative values) so load raw labeled data
# and build a separate pipeline with only non-negative columns
# also filter out any negative CRSElapsedTime rows - raw data has some bad values
print('Training Naive Bayes...')

df_raw = spark.read.parquet(str(DATA_DIR / 'labeled'))

from pyspark.sql.types import DoubleType

# only columns guaranteed to be >= 0, and filter out any negatives
NB_FEATURES = ['Month', 'DayOfWeek', 'Distance', 'CRSElapsedTime']

df_nb = (
    df_raw.select(
        F.col('Month').cast(DoubleType()),
        F.col('DayOfWeek').cast(DoubleType()),
        F.col('Distance').cast(DoubleType()),
        F.col('CRSElapsedTime').cast(DoubleType()),
        F.col('label')
    )
    .na.drop()
    .filter(F.col('CRSElapsedTime') > 0)
    .filter(F.col('Distance') > 0)
    .filter(F.col('Month') > 0)
    .filter(F.col('DayOfWeek') > 0)
)

nb_train, nb_test = df_nb.randomSplit([0.8, 0.2], seed=42)

nb_assembler = VectorAssembler(inputCols=NB_FEATURES, outputCol='nb_features', handleInvalid='skip')
nb = NaiveBayes(featuresCol='nb_features', labelCol='label', modelType='multinomial')
nb_pipeline = Pipeline(stages=[nb_assembler, nb])

nb_grid = ParamGridBuilder().addGrid(nb.smoothing, [0.5, 1.0, 2.0]).build()

nb_cv = CrossValidator(
    estimator=nb_pipeline, estimatorParamMaps=nb_grid,
    evaluator=auc_roc_eval, numFolds=3, seed=42
)

t0 = time.time()
nb_model = nb_cv.fit(nb_train)
nb_time  = time.time() - t0

nb_preds  = nb_model.transform(nb_test)
nb_auc    = auc_roc_eval.evaluate(nb_preds)
nb_auc_pr = auc_pr_eval.evaluate(nb_preds)
nb_f1     = f1_eval.evaluate(nb_preds)

results['Naive Bayes'] = {
    'auc_roc': nb_auc, 'auc_pr': nb_auc_pr, 'f1': nb_f1, 'train_sec': nb_time
}

print(f'Train time : {nb_time:.1f}s')
print(f'AUC-ROC    : {nb_auc:.4f}')
print(f'AUC-PR     : {nb_auc_pr:.4f}')
print(f'F1         : {nb_f1:.4f}')

Training Naive Bayes...
Train time : 97.2s
AUC-ROC    : 0.4722
AUC-PR     : 0.1958
F1         : 0.6968


In [7]:
# NB confusion matrix and model.pi (class log priors) as in the Week 3 lab
print('Naive Bayes — Confusion Matrix')
nb_preds.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()

best_nb = nb_model.bestModel.stages[-1]
print('Class log priors (model.pi):', best_nb.pi)

nb_preds.select('label', 'prediction', 'probability') \
    .write.mode('overwrite').parquet(str(DATA_DIR / 'predictions_nb'))

Naive Bayes — Confusion Matrix
+-----+----------+-------+
|label|prediction|  count|
+-----+----------+-------+
|  0.0|       0.0|1794672|
|  0.0|       1.0| 218762|
|  1.0|       0.0| 467815|
|  1.0|       1.0|  63714|
+-----+----------+-------+

Class log priors (model.pi): [-0.23468447470431464,-1.5645617507845788]


## Model 3 — Random Forest

In [ ]:
# Model 3: Random Forest
# ensemble of trees with random feature subsets
# numTrees and maxDepth in the param grid - same approach as NYC Taxi lab activity
# Note: using a 30% sample for CV to avoid driver OOM on 10M+ rows
print('Training Random Forest...')

rf_train = train_df.sample(fraction=0.3, seed=42)
rf_train.persist(StorageLevel.DISK_ONLY)
print(f'RF train sample rows: {rf_train.count():,}')

rf = RandomForestClassifier(featuresCol='features', labelCol='label', seed=42)

rf_grid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [10, 20])
    .addGrid(rf.maxDepth, [3, 5])
    .build()
)

rf_cv = CrossValidator(
    estimator=rf, estimatorParamMaps=rf_grid,
    evaluator=auc_roc_eval, numFolds=3, seed=42
)

t0 = time.time()
rf_model = rf_cv.fit(rf_train)
rf_time  = time.time() - t0

rf_preds  = rf_model.transform(test_df)
rf_auc    = auc_roc_eval.evaluate(rf_preds)
rf_auc_pr = auc_pr_eval.evaluate(rf_preds)
rf_f1     = f1_eval.evaluate(rf_preds)

results['Random Forest'] = {
    'auc_roc': rf_auc, 'auc_pr': rf_auc_pr, 'f1': rf_f1, 'train_sec': rf_time
}

print(f'Train time : {rf_time:.1f}s')
print(f'AUC-ROC    : {rf_auc:.4f}')
print(f'AUC-PR     : {rf_auc_pr:.4f}')
print(f'F1         : {rf_f1:.4f}')

In [ ]:
# RF confusion matrix and feature importances
print('Random Forest — Confusion Matrix')
rf_preds.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()

best_rf = rf_model.bestModel
importances = best_rf.featureImportances.toArray()

FEATURE_COLS = [
    'Month', 'DayOfWeek', 'hour_of_day',
    'CRSElapsedTime', 'Distance',
    'airline_idx', 'origin_idx', 'dest_idx'
]

print('Feature importances (Random Forest):')
for feat, imp in sorted(zip(FEATURE_COLS, importances), key=lambda x: -x[1]):
    print(f'  {feat:<20} {imp:.4f}')

np.save(str(DATA_DIR / 'rf_importances.npy'), importances)
rf_preds.select('label', 'prediction', 'probability') \
    .write.mode('overwrite').parquet(str(DATA_DIR / 'predictions_rf'))

## Model 4 — Gradient Boosted Trees (GBT)

In [ ]:
# Model 4: Gradient Boosted Trees
# sequential ensemble - maxIter and stepSize in the param grid
# using same 30% sample as RF to keep memory stable
print('Training GBT (this will take a while)...')

gbt_train = train_df.sample(fraction=0.3, seed=42)
gbt_train.persist(StorageLevel.DISK_ONLY)

gbt = GBTClassifier(featuresCol='features', labelCol='label', seed=42)

gbt_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxIter,  [10, 20])
    .addGrid(gbt.stepSize, [0.1, 0.05])
    .build()
)

gbt_cv = CrossValidator(
    estimator=gbt, estimatorParamMaps=gbt_grid,
    evaluator=auc_roc_eval, numFolds=3, seed=42
)

t0 = time.time()
gbt_model = gbt_cv.fit(gbt_train)
gbt_time  = time.time() - t0

gbt_preds  = gbt_model.transform(test_df)
gbt_auc    = auc_roc_eval.evaluate(gbt_preds)
gbt_auc_pr = auc_pr_eval.evaluate(gbt_preds)
gbt_f1     = f1_eval.evaluate(gbt_preds)

results['GBT'] = {
    'auc_roc': gbt_auc, 'auc_pr': gbt_auc_pr, 'f1': gbt_f1, 'train_sec': gbt_time
}

print(f'Train time : {gbt_time:.1f}s')
print(f'AUC-ROC    : {gbt_auc:.4f}')
print(f'AUC-PR     : {gbt_auc_pr:.4f}')
print(f'F1         : {gbt_f1:.4f}')

In [ ]:
# GBT confusion matrix and feature importances
print('GBT — Confusion Matrix')
gbt_preds.groupBy('label', 'prediction').count().orderBy('label', 'prediction').show()

best_gbt = gbt_model.bestModel
gbt_importances = best_gbt.featureImportances.toArray()

print('Feature importances (GBT):')
for feat, imp in sorted(zip(FEATURE_COLS, gbt_importances), key=lambda x: -x[1]):
    print(f'  {feat:<20} {imp:.4f}')

np.save(str(DATA_DIR / 'gbt_importances.npy'), gbt_importances)
gbt_preds.select('label', 'prediction', 'probability') \
    .write.mode('overwrite').parquet(str(DATA_DIR / 'predictions_gbt'))

## Summary Comparison Table

In [ ]:
# all-model comparison table - spark.createDataFrame approach from NYC Taxi lab activity
summary_rows = [
    (name, r['auc_roc'], r['auc_pr'], r['f1'], r['train_sec'])
    for name, r in results.items()
]

summary_df = spark.createDataFrame(
    summary_rows,
    ['model', 'auc_roc', 'auc_pr', 'f1', 'train_sec']
)

summary_df.orderBy('auc_roc', ascending=False).show(truncate=False)

best_name = max(results, key=lambda k: results[k]['auc_roc'])
print(f'Best model by AUC-ROC: {best_name}  ({results[best_name]["auc_roc"]:.4f})')

In [ ]:
# Full metrics table including accuracy, precision, recall
# computed from the saved prediction parquet files
accuracy_eval  = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='accuracy')
precision_eval = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedPrecision')
recall_eval    = MulticlassClassificationEvaluator(labelCol='label', predictionCol='prediction', metricName='weightedRecall')

pred_paths = {
    'Logistic Regression': str(DATA_DIR / 'predictions_lr'),
    'Naive Bayes':         str(DATA_DIR / 'predictions_nb'),
    'Random Forest':       str(DATA_DIR / 'predictions_rf'),
    'GBT':                 str(DATA_DIR / 'predictions_gbt'),
}

full_rows = []
for name, path in pred_paths.items():
    p   = spark.read.parquet(path)
    acc = accuracy_eval.evaluate(p)
    pr  = precision_eval.evaluate(p)
    rec = recall_eval.evaluate(p)
    full_rows.append((
        name,
        round(results[name]['auc_roc'],  4),
        round(results[name]['auc_pr'],   4),
        round(results[name]['f1'],       4),
        round(acc, 4),
        round(pr,  4),
        round(rec, 4),
        round(results[name]['train_sec'], 1),
    ))

full_df = spark.createDataFrame(
    full_rows,
    ['model', 'auc_roc', 'auc_pr', 'f1', 'accuracy', 'precision', 'recall', 'train_sec']
)
print('=== Full Model Metrics Table ===')
full_df.orderBy('auc_roc', ascending=False).show(truncate=False)

## 3.3 Model Comparison and Training Time Analysis

GBT did the best overall in predictive performance with an AUC-ROC of 0.6595 and an AUC-PR of 0.3376 – both of which were the highest among the four models. This supports the original hypothesis that there are non-linear relationships between the airline, hour, month, and route features that drive flight delays. A boosting model such as GBT is well suited to identify these non-linear patterns, learning them sequentially without being bound by a linear decision boundary, unlike other models.

Logistic Regression (AUC-ROC 0.621) followed GBT and provided a competitive linear baseline, with a relatively good level of interpretability. As expected for an imbalanced dataset (with 21% positive rate), no class weights were set and, therefore, the confusion matrix and model predicted that the flight was "not delayed" – which it should not, given this bias. However, an AUC-ROC score of 0.621 shows that the features used have some linear power. Logistic Regression models also produced coefficients directly relevant to later analysis on model interpretability.

Random Forest (AUC-ROC 0.500) result indicated that the model performed just about equivalent to random guessing. In an effort to prevent the Spark driver from running out of memory due to over-exhaustion (i.e., having too many trees/ too much tree depth), only 10–20 trees were used, and the maximum depth of the tree was set at 3–5. The entire dataset was subsampled, reducing the training set size to just 30% of the original training data, to again mitigate memory issues.

Because of these limitations, meaningful splits were not made and the importance of every single feature worked out to be 0.000.

GBT did considerably better than Random Forest under the same conditions, illustrating the power of a boosting approach on a highly imbalanced dataset in using each weak learner efficiently.

Naive Bayes (AUC-ROC 0.473) rounded out the bunch, producing a result lower than random – as expected, since this method is unable to account for the non-independence between features (airline, hour, month), which definitely interact in how flight delays happen. Furthermore, Naive Bayes is utilising only 4 raw features as opposed to the 8 scaled features given to other models. Naive Bayes offers no functionality to leverage feature interactions since its calculations are solely centred around estimating independent class proportions per feature, but it gives us a floor on where we should expect results, making it useful for reference.

Training time has a correlation with what you'd expect. LR (156s) and NB (89s) were the fastest – both solve simple problems – whereas the RF (78s) is just barely higher as it utilises limited data on narrow (3–5 tree depth, 10–20 tree count) shallow trees. However, GBT (235s) takes the longest to complete — this is because in order to learn efficiently, boosting requires its trees to be dependent (building one tree off the previous iteration). Additionally, the addition of CrossValidator (12 full model fits) makes it considerably slower.

Though it costs the most, GBT offers the most predictive output and is the model most suitable for Task 4 and Task 5 evaluation on this dataset.

In [ ]:
# save best model and summary for Task 5
best_model_map = {
    'Logistic Regression': lr_model.bestModel,
    'Random Forest':       rf_model.bestModel,
    'GBT':                 gbt_model.bestModel,
}

if best_name in best_model_map:
    best_model_map[best_name].write().overwrite().save(str(DATA_DIR / 'best_model'))
    print(f'Best model saved: {best_name}')

with open(str(DATA_DIR / 'model_summary.json'), 'w') as f:
    json.dump(results, f, indent=2)

print('Summary saved to model_summary.json')
print('Task 3 complete')